In [2]:
!git clone https://github.com/AI4Bharat/IndicLID.git
!pip install fasttext

fatal: destination path 'IndicLID' already exists and is not an empty directory.


In [3]:
# Move into the IndicLID directory
%cd IndicLID/Inference/ai4bharat

# Create the models directory structure
!mkdir -p models

# Download and unzip the models
!wget https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftn.zip
!unzip indiclid-ftn.zip -d models/

!wget https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftr.zip
!unzip indiclid-ftr.zip -d models/

!wget https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-bert.zip
!unzip indiclid-bert.zip -d models/

# Clean up the zip files
!rm *.zip

/content/IndicLID/Inference/ai4bharat
--2026-09-13 15:09:29--  https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftn.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/605931363/8141193b-39cd-4c36-9dec-576afa53c18f?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-13T15%3A55%3A07Z&rscd=attachment%3B+filename%3Dindiclid-ftn.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-09-13T14%3A54%3A24Z&ske=2026-09-13T15%3A55%3A07Z&sks=b&skv=2018-11-09&sig=cwcjYKQU8tGXNMAAlNOvabrAPDmKtqNMwLzCql2RqzM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4OTMxNTc2OSwibmJmIjoxNzg5MzEyMTY5LCJwYXRoI

In [10]:
import sys
import torch
import pandas as pd
sys.path.append('.')
from IndicLID import IndicLID
from transformers import BertConfig, BertForSequenceClassification

# --- 1. THE TOKENIZER PATCH ---
def patched_IndicBERT_roman_inference(self, IndicLID_BERT_inputs, output_dict, batch_size):
    if not IndicLID_BERT_inputs:
        return output_dict

    df = pd.DataFrame(IndicLID_BERT_inputs)
    dataloader = self.get_dataloaders(df.iloc[:,0], df.iloc[:,1], batch_size)

    with torch.no_grad():
        for data in dataloader:
            batch_indices = data[0]
            batch_inputs = data[1]

            word_embeddings = self.IndicLID_BERT_tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True, max_length=512)
            word_embeddings = word_embeddings.to(self.device)

            token_type_ids = word_embeddings.get('token_type_ids', torch.zeros_like(word_embeddings['input_ids']))

            batch_outputs = self.IndicLID_BERT(
                word_embeddings['input_ids'],
                token_type_ids=token_type_ids,
                attention_mask=word_embeddings['attention_mask']
            )

            _, batch_predicted = torch.max(batch_outputs.logits, 1)

            for index, input, pred_label, logit in zip(batch_indices, batch_inputs, batch_predicted, batch_outputs.logits):
                output_dict[index] = (input,
                                        self.IndicLID_lang_code_dict_reverse[pred_label.item()],
                                        logit[pred_label.item()].item(), 'IndicLID-BERT'
                                        )
    return output_dict

IndicLID.IndicBERT_roman_inference = patched_IndicBERT_roman_inference

# --- 2. LOAD LEGACY MODEL ---
print("Loading legacy IndicLID model...")
model = IndicLID(input_threshold=0.5, roman_lid_threshold=0.6)

# --- 3. THE ULTIMATE UPGRADE PATCH ---
print("Upgrading model architecture to modern transformers...")
# Extract the raw tensors (weights) from the broken legacy model
state_dict = model.IndicLID_BERT.state_dict()
# Extract the old config as a simple dictionary
config_dict = model.IndicLID_BERT.config.to_dict()

# Create a brand new modern model using the old config values
new_config = BertConfig(**config_dict)
fresh_model = BertForSequenceClassification(new_config)

# Load the legacy weights into the modern model structure
# strict=False tells PyTorch to safely ignore the old 2023 variables like 'position_ids'
fresh_model.load_state_dict(state_dict, strict=False)

fresh_model.to(model.device)
fresh_model.eval()

# Replace the broken model with the fresh one!
model.IndicLID_BERT = fresh_model
print("Upgrade complete!")

# --- 4. TEST ---
test_text = "Njan innu busy aanu"
print(f"\nForcing BERT inference on: '{test_text}'...")

result = model.IndicBERT_roman_inference([(0, test_text)], {}, 1)

print("Result:", result)

Loading legacy IndicLID model...
Upgrading model architecture to modern transformers...
Upgrade complete!

Forcing BERT inference on: 'Njan innu busy aanu'...
Result: {tensor(0): ('Njan innu busy aanu', 'mal_Latn', 3.9314937591552734, 'IndicLID-BERT')}
